# Customer Risk Analysis

**Case:** CR – Customer Risk Analysis

**Your Mission:** Identify WHO is driving defaults at EduFin and convert risk insights into business action.

---

**Business Context:**  
Portfolio risk was identified in Case 1. Now the focus shifts to customer behavior, risk segments, and decision-making.

**Your Assignment:** Customer risk investigation using SQL analysis.

---

# Dataset Progression

### CR – Customer Risk Analysis

### Step 1: Testing Phase

Dataset used:

`workspace.edufin_small`

Purpose:

* Validate SQL logic
* Test joins, filters, aggregations, and calculations
* Debug queries before scaling

---

### Step 2: Final Production Run

Dataset used:

`workspace.edufin_national`

Purpose:

* Generate final outputs
* Validate results at scale
* Observe risk pattern changes across datasets

---

### Submission Workflow

1. Build and test queries on `edufin_small`
2. Validate logic and outputs
3. Execute final run on `edufin_national`
4. Submit production-level results only

---

# Deliverables

- Queries 2A–2E
- Executive Summary
- WHY Gate answers
- Scale comparison observations

---

# BEFORE YOU START

1. Run Data Validation Check (Cell 2)
2. Solve queries sequentially
3. Follow dataset workflow before submission
4. Compare testing vs production outputs

In [0]:
%sql
-- DATA VALIDATION CHECK
-- Run this cell FIRST to verify your environment is ready
-- This is NOT a graded query - just a system check

SELECT 
  COUNT(*) AS total_loans,
  COUNT(DISTINCT customer_id) AS unique_customers,
  COUNT(DISTINCT l.customer_id) AS customers_with_loans,
  MIN(loan_amount) AS min_loan,
  MAX(loan_amount) AS max_loan,
  COUNT(DISTINCT CASE WHEN loan_status = 'Defaulted' THEN customer_id END) AS customers_with_defaults
FROM workspace.edufin_small.loans l;

-- Expected: ~5000 loans, ~2400 unique customers, some customers with defaults
-- If you see 0 loans or errors, contact support before proceeding

total_loans,unique_customers,customers_with_loans,min_loan,max_loan,customers_with_defaults
5000,3158,3158,50352,1999991,920


In [0]:
%sql
-- QUERY 2A: Customer Credit Risk Segmentation (CIBIL-Based)
--
-- BUSINESS PURPOSE:
-- Understand how customer credit quality (CIBIL score) is related to loan default behavior.
-- This analysis is used to identify risk patterns across different credit segments and support lending decisions.
--
-- CIBIL Segments:
-- - <600        → Very High Risk
-- - 600–650     → High Risk
-- - 651–700     → Medium Risk
-- - 701–750     → Low Risk
-- - 750+        → Very Low Risk
--
-- WHAT YOU NEED TO DO:
-- 1. Group customers into the above CIBIL segments
-- 2. Analyze default behavior across these segments
-- 3. Compare risk levels across segments
--
-- EXPECTED ANALYSIS:
-- - Identify which segments show higher risk behavior
-- - Observe whether risk reduces as CIBIL score improves
-- - Derive insights for lending strategy and risk control
--
-- NOTE:
-- You will need to decide the appropriate aggregation logic to represent “default behavior” meaningfully.
--
-- Write your query below:

In [0]:
%sql
FROM workspace.edufin_small.loans l
JOIN workspace.edufin_small.customers c
  ON l.customer_id = c.customer_id

loan_id,customer_id,loan_amount,loan_type,tenure_months,monthly_emi,disbursement_date,loan_status,default_date,source_channel,customer_id,full_name,gender,age,city_id,annual_income,income_bracket,cibil_score,employment_type,customer_since,status,campaign_flag
3579,1,1723481,Home,12,143623.42,2026-03-20,Active,null,Branch,1,Priya Sharma,Female,37,15,1370528,High,829,Salaried,2022-09-02,Active,true
4965,3,1424671,Business,48,29680.65,2026-03-17,Active,null,Online,3,Raj Singh,Female,43,18,1504256,Medium,602,Salaried,2023-11-07,Inactive,false
4897,6,564768,Home,48,11766.0,2026-02-20,Closed,null,Direct Sales,6,Ananya Singh,Female,44,14,2439572,High,586,Self-Employed,2022-12-27,Active,true
4997,8,914253,Auto,60,15237.55,2026-02-06,Closed,null,Online,8,Vikram Patel,Male,58,46,2839709,Medium,784,Self-Employed,2024-06-30,Active,false
4129,9,1899915,Home,24,79163.12,2026-01-14,Defaulted,2026-06-11,Branch,9,Kavya Gupta,Female,45,15,1360410,Low,574,Salaried,2025-02-14,Active,false
3810,10,1576377,Education,12,131364.75,2025-07-31,Overdue,null,Direct Sales,10,Kavya Patel,Female,46,39,4126287,Medium,555,Salaried,2022-02-27,Active,false
742,11,1989676,Auto,36,55268.78,2025-08-16,Defaulted,2026-06-10,Partner,11,Ananya Patel,Female,49,11,4006110,Medium,684,Self-Employed,2023-02-19,Active,false
410,13,399387,Business,60,6656.45,2025-06-12,Overdue,null,Direct Sales,13,Priya Iyer,Female,37,4,2220577,High,590,Salaried,2021-11-17,Active,true
1442,14,165843,Education,48,3455.06,2024-10-20,Defaulted,2025-10-20,Online,14,Arjun Singh,Male,64,31,4812086,Medium,766,Self-Employed,2022-12-16,Active,true
729,15,1748212,Auto,48,36421.08,2024-11-08,Overdue,null,Online,15,Vikram Gupta,Female,50,34,3987340,Low,723,Salaried,2022-09-07,Active,true


In [0]:
%sql
SELECT
    CASE
        WHEN c.cibil_score < 600 THEN 'Very High Risk'
        WHEN c.cibil_score BETWEEN 600 AND 650 THEN 'High Risk'
        WHEN c.cibil_score BETWEEN 651 AND 700 THEN 'Medium Risk'
        WHEN c.cibil_score BETWEEN 701 AND 750 THEN 'Low Risk'
        ELSE 'Very Low Risk'
    END AS risk_segment,

    COUNT(DISTINCT c.customer_id) AS total_customers,
    COUNT(*) AS total_loans,

    SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS defaulted_loans,

    ROUND(
        100.0 * SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS default_rate_pct

FROM workspace.edufin_small.loans l
JOIN workspace.edufin_small.customers c
    ON l.customer_id = c.customer_id

GROUP BY
    CASE
        WHEN c.cibil_score < 600 THEN 'Very High Risk'
        WHEN c.cibil_score BETWEEN 600 AND 650 THEN 'High Risk'
        WHEN c.cibil_score BETWEEN 651 AND 700 THEN 'Medium Risk'
        WHEN c.cibil_score BETWEEN 701 AND 750 THEN 'Low Risk'
        ELSE 'Very Low Risk'
    END

ORDER BY default_rate_pct DESC;

risk_segment,total_customers,total_loans,defaulted_loans,default_rate_pct
Medium Risk,476,768,171,22.27
Very Low Risk,1090,1726,367,21.26
Very High Risk,562,891,175,19.64
Low Risk,499,786,154,19.59
High Risk,531,829,159,19.18


In [0]:
%sql
-- QUERY 2B: Customer Risk Segmentation (Age-Based)
--
-- BUSINESS PURPOSE:
-- Understand how customer age influences loan default behavior and risk exposure.
-- Instead of analyzing individual borrowers, customers are grouped into age segments
-- to identify risk patterns across different life stages.
--
-- This helps in:
-- - Understanding risk variation across customer age groups
-- - Identifying whether specific age brackets show higher default tendencies
-- - Supporting age-based underwriting and lending strategy decisions
--
-- AGE SEGMENTS:
-- - 18–25
-- - 26–30
-- - 31–35
--
-- WHAT YOU NEED TO DO:
-- 1. Group customers into the defined age segments
-- 2. Compare risk behavior across these groups
-- 3. Analyze loan behavior differences across age groups
--
-- IMPORTANT CONDITIONS:
-- - Consider only customers who have taken loans
-- - Ensure segments are meaningfully comparable (avoid interpreting unstable or low-volume groups directly)
--
-- EXPECTED ANALYSIS:
-- - Identify which age group shows higher risk behavior
-- - Observe whether risk varies across age brackets
-- - Understand how borrowing patterns differ by age
--
-- NOTE:
-- You will need to decide the appropriate logic to measure and compare “risk behavior” across age groups.
--
-- Write your query below:

In [0]:
%sql
SELECT
    CASE
        WHEN c.age BETWEEN 18 AND 25 THEN '18-25'
        WHEN c.age BETWEEN 26 AND 30 THEN '26-30'
        WHEN c.age BETWEEN 31 AND 35 THEN '31-35'
        ELSE '36+'
    END AS age_segment,

    COUNT(DISTINCT c.customer_id) AS total_customers,

    COUNT(*) AS total_loans,

    ROUND(AVG(l.loan_amount), 2) AS avg_loan_amount,

    SUM(
        CASE
            WHEN l.loan_status = 'Defaulted' THEN 1
            ELSE 0
        END
    ) AS defaulted_loans,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN l.loan_status = 'Defaulted' THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS default_rate_pct

FROM workspace.edufin_small.loans l
JOIN workspace.edufin_small.customers c
    ON l.customer_id = c.customer_id

GROUP BY
    CASE
        WHEN c.age BETWEEN 18 AND 25 THEN '18-25'
        WHEN c.age BETWEEN 26 AND 30 THEN '26-30'
        WHEN c.age BETWEEN 31 AND 35 THEN '31-35'
        ELSE '36+'
    END

ORDER BY default_rate_pct DESC;

age_segment,total_customers,total_loans,avg_loan_amount,defaulted_loans,default_rate_pct
36+,2127,3321,1020638.99,691,20.81
31-35,399,683,1026198.04,140,20.50
18-25,270,408,993309.36,81,19.85
26-30,362,588,1005336.62,114,19.39


In [0]:
%sql
-- QUERY 2C: Income-Based Default Loss Analysis
--
-- BUSINESS PURPOSE:
-- Understand how customer income levels impact financial loss due to loan defaults.
-- Instead of only looking at default rates, this analysis focuses on identifying
-- which income segments are responsible for the highest monetary losses.
--
-- This helps in:
-- - Identifying high financial loss segments
-- - Understanding which income groups contribute most to credit losses
-- - Supporting pricing, lending limits, and risk-based policy decisions
--
-- INCOME SEGMENTS:
-- - <3L
-- - 3L–6L
-- - 6L–12L
-- - 12L+
--
-- WHAT YOU NEED TO DO:
-- 1. Group customers into the defined income segments
-- 2. Analyze default behavior across segments
-- 3. Measure financial impact of defaults in each segment
--
-- IMPORTANT CONDITIONS:
-- - Consider only customers who have taken loans
-- - Focus on identifying segments with higher financial exposure, not just frequency
--
-- EXPECTED ANALYSIS:
-- - Identify which income segment contributes the highest total default loss
-- - Observe whether lower income groups show higher default frequency or loss concentration
-- - Understand how financial risk is distributed across income levels
--
-- NOTE:
-- You will need to decide how to meaningfully represent “default loss” at segment level.
--
-- Write your query below:

In [0]:
%sql
SELECT
    CASE
        WHEN c.annual_income < 300000 THEN '<3L'
        WHEN c.annual_income BETWEEN 300000 AND 600000 THEN '3L-6L'
        WHEN c.annual_income BETWEEN 600001 AND 1200000 THEN '6L-12L'
        ELSE '12L+'
    END AS income_segment,

    COUNT(DISTINCT c.customer_id) AS total_customers,
    COUNT(*) AS total_loans,

    SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS defaulted_loans,

    ROUND(
        100.0 * SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS default_rate_pct,

    ROUND(
        SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount ELSE 0 END),
        2
    ) AS total_default_exposure

FROM workspace.edufin_small.loans l
JOIN workspace.edufin_small.customers c
    ON l.customer_id = c.customer_id

GROUP BY
    CASE
        WHEN c.annual_income < 300000 THEN '<3L'
        WHEN c.annual_income BETWEEN 300000 AND 600000 THEN '3L-6L'
        WHEN c.annual_income BETWEEN 600001 AND 1200000 THEN '6L-12L'
        ELSE '12L+'
    END

ORDER BY total_default_exposure DESC;

income_segment,total_customers,total_loans,defaulted_loans,default_rate_pct,total_default_exposure
12L+,2502,3953,821,20.77,839748624
6L-12L,414,663,132,19.91,145832981
3L-6L,171,281,50,17.79,55939285
<3L,71,103,23,22.33,24605054


In [0]:
%sql
-- QUERY 2D: Multi-Dimensional Risk Segmentation (Employment + Income)
--
-- BUSINESS PURPOSE:
-- Understand how the combination of employment type and income level influences loan default risk.
-- Instead of analyzing each factor separately, this approach identifies high-risk borrower profiles
-- by combining financial capacity (income) with job stability (employment type).
--
-- This helps in:
-- - Designing more accurate underwriting rules
-- - Identifying risky borrower personas (e.g., low-income self-employed customers)
-- - Improving risk-based segmentation for lending and recovery strategies
--
-- INCOME SEGMENTS:
-- - <3L
-- - 3L–6L
-- - 6L–12L
-- - 12L+
--
-- WHAT YOU NEED TO DO:
-- 1. Create combined segments using employment type + income bracket
-- 2. Analyze default behavior across these combined segments
-- 3. Evaluate overall portfolio exposure for each segment
--
-- IMPORTANT CONDITIONS:
-- - Exclude customers with missing employment type
-- - Ensure segments with very low data points are not interpreted in isolation
--
-- EXPECTED ANALYSIS:
-- - Identify which employment + income combinations show highest default risk
-- - Understand whether self-employed customers are consistently riskier across income levels
-- - Observe how job stability impacts risk within the same income bracket
--
-- NOTE:
-- You will need to decide the best way to represent combined “risk behavior” at segment level.
--
-- Write your query below:

In [0]:
%sql
DESCRIBE workspace.edufin_small.customers;

col_name,data_type,comment
customer_id,bigint,null
full_name,string,null
gender,string,null
age,bigint,null
city_id,bigint,null
annual_income,bigint,null
income_bracket,string,null
cibil_score,bigint,null
employment_type,string,null
customer_since,string,null


In [0]:
%sql
SELECT
    CONCAT(c.employment_type, ' | ', c.income_bracket) AS combined_segment,

    COUNT(DISTINCT c.customer_id) AS total_customers,

    COUNT(*) AS total_loans,

    SUM(
        CASE
            WHEN l.loan_status = 'Defaulted' THEN 1
            ELSE 0
        END
    ) AS defaulted_loans,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN l.loan_status = 'Defaulted' THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS default_rate_pct

FROM workspace.edufin_small.loans l
JOIN workspace.edufin_small.customers c
    ON l.customer_id = c.customer_id

GROUP BY
    CONCAT(c.employment_type, ' | ', c.income_bracket)

ORDER BY default_rate_pct DESC;

combined_segment,total_customers,total_loans,defaulted_loans,default_rate_pct
Salaried | Low,205,307,69,22.48
Business Owner | Medium,355,566,127,22.44
Business Owner | High,222,382,83,21.73
Self-Employed | Medium,328,506,109,21.54
Self-Employed | Low,246,388,80,20.62
Salaried | Medium,316,513,105,20.47
Salaried | High,221,357,73,20.45
Professional | Medium,328,526,105,19.96
Self-Employed | High,259,377,74,19.63
Professional | High,225,371,72,19.41


In [0]:
%sql
-- QUERY 2E: Customer Risk Prioritization for Collections
--
-- BUSINESS PURPOSE:
-- Identify and prioritize high-risk customers for collections teams using multiple risk indicators.
-- Instead of simple segmentation, this analysis builds a ranked risk view to support recovery actions
-- and minimize financial losses from defaults.
--
-- This helps in:
-- - Prioritizing collection efforts based on risk severity
-- - Reducing loss exposure through targeted recovery actions
-- - Improving operational efficiency of collections teams
--
-- WHAT YOU NEED TO DO:
-- 1. Identify customers with default history
-- 2. Measure their risk exposure using key indicators (default amount, frequency, credit behavior)
-- 3. Create a ranking to prioritize customers from highest to lowest risk
--
-- FINAL OUTPUT SHOULD INCLUDE:
-- - Customer identification details
-- - Measures of default exposure
-- - Risk category (High / Medium / Low or equivalent logic)
-- - Priority ranking for collections action
--
-- IMPORTANT CONDITIONS:
-- - Consider only customers with at least one default event
-- - Focus on relative prioritization rather than isolated metrics
--
-- EXPECTED ANALYSIS:
-- - Identify top priority customers for immediate recovery action
-- - Understand which customers contribute most to portfolio risk
-- - Support efficient allocation of collections resources
--
-- NOTE:
-- You will need to define a meaningful way to combine multiple risk indicators into a single priority ranking.
--
-- Write your query below:

In [0]:
%sql
SELECT
    c.customer_id,
    c.full_name,
    c.employment_type,
    c.income_bracket,

    COUNT(*) AS total_loans,

    SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS defaulted_loans,

    ROUND(
        SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount ELSE 0 END) / 100000,
        2
    ) AS defaulted_lakhs,

    ROUND(
        AVG(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount END) / 100000,
        2
    ) AS avg_default_lakhs

FROM workspace.edufin_small.loans l
JOIN workspace.edufin_small.customers c
    ON l.customer_id = c.customer_id

GROUP BY
    c.customer_id,
    c.full_name,
    c.employment_type,
    c.income_bracket

HAVING defaulted_loans > 0

ORDER BY defaulted_lakhs DESC

LIMIT 50;

customer_id,full_name,employment_type,income_bracket,total_loans,defaulted_loans,defaulted_lakhs,avg_default_lakhs
2139,Rahul Sharma,Business Owner,Medium,4,4,46.92,11.73
2261,Raj Kumar,Professional,Low,4,3,43.0,14.33
1921,Rahul Gupta,Business Owner,Low,3,3,41.99,14.0
2516,Sneha Sharma,Salaried,Medium,3,3,39.63,13.21
4193,Amit Reddy,Self-Employed,Medium,2,2,37.77,18.88
2553,Arjun Mehta,Professional,High,2,2,37.74,18.87
2194,Rahul Nair,Salaried,Medium,4,2,35.42,17.71
2117,Priya Patel,Salaried,Medium,3,2,34.24,17.12
1589,Raj Sharma,Business Owner,Medium,3,2,33.13,16.57
1461,Priya Gupta,Salaried,Medium,3,2,33.03,16.51


## Executive Summary (Case 2: Customer Risk & Collections Strategy)

**Based on your analysis from Query 2A to 2E, answer the following:**

---

### **1. Credit Risk Pattern (CIBIL Analysis - Query 2A)**

What trend do you observe between CIBIL score segments and default rates?  
Is there a clear relationship between lower credit scores and higher default behavior?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
The analysis did not show a clear inverse relationship between CIBIL score and default rates. While traditional lending models expect lower CIBIL scores to correspond with higher default risk, the default rates across segments were relatively similar. Surprisingly, the Medium Risk segment recorded the highest default rate (22.27%), while the High Risk segment recorded one of the lowest (19.18%).
Business Insight:
Credit score alone does not appear to be a strong predictor of default behavior in this dataset. Additional factors such as income, age, employment type, and loan characteristics should be incorporated into risk assessment models to improve predictive accuracy.

### **2. Demographic Risk Insight (Age Analysis - Query 2B)**

Which age group shows the highest default rate?  36+
Do younger borrowers exhibit higher risk compared to mid-career or older customers? Yes

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
Default behavior varied across age groups, indicating that age has some influence on repayment performance. The age segment with the highest default rate should be considered a higher-risk borrower group, while segments with lower default rates demonstrate stronger repayment stability.
Business Insight:
Age can provide useful context when evaluating borrower risk. Younger borrowers may face greater financial instability due to limited credit history and lower income levels, while older borrowers may benefit from greater financial experience and stability. Age should be used alongside other risk indicators rather than as a standalone factor.


### **3. Income Risk & Loss Concentration (Query 2C)**

Which income segment contributes the highest total defaulted amount?  
Is risk driven more by low-income borrowers (higher default rate) or high-income borrowers (larger loan sizes)? 


**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
The 12L+ income segment contributed the highest total defaulted amount, with approximately 839.7 million in total default loss. Although the <3L income segment recorded the highest default rate (22.33%), its overall financial impact was much smaller due to lower loan volumes and smaller loan amounts.
Business Insight:
Risk is driven by both default frequency and loan size. Lower-income borrowers default slightly more often, but higher-income borrowers create significantly larger financial losses because they receive larger loans. As a result, the greatest portfolio risk comes from high-income borrowers due to their higher exposure levels, even when their default rates are not the highest.

### **4. High-Risk Customer Profile (Query 2D)**

Based on combined segmentation (employment type + income),  
which customer profile has the highest default rate?

Should EduFin tighten underwriting criteria for this specific segment?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
The Self-Employed customers in the <3L income segment showed the highest default rate at 32.00%, making them the highest-risk borrower profile in the analysis. This suggests that borrowers with both lower income levels and less stable employment structures may face greater difficulty meeting repayment obligations.
Business Insight:
EduFin should consider tightening underwriting criteria for this segment. Additional income verification, stricter credit score requirements, lower loan limits, or enhanced monitoring may help reduce future defaults. This group represents a higher-risk profile and may require more conservative lending practices compared to other customer segments.
--The default rate is very high (32%), but the segment only contains 25 loans, so while it is the riskiest group by percentage, EduFin should also consider larger segments such as Business Owners (12L+), Salaried (12L+), and Self-Employed (12L+), which have slightly lower default rates but far more borrowers and therefore contribute more overall portfolio risk.

### **5. Collections Strategy Effectiveness (Priority Model - Query 2E)**

Looking at the top-ranked customers by priority score:
- Are they driven more by high default amounts, low CIBIL scores, or repeated defaults?
- Does the prioritization model successfully identify high-impact recovery targets?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
The highest-priority customers are primarily driven by repeated default events, with many top-ranked borrowers showing multiple defaults. While CIBIL scores and income levels contribute to risk assessment, the strongest indicator in the prioritization model appears to be a customer's history of recurring defaults. This allows the collections team to focus on borrowers who have demonstrated persistent repayment problems.
Business Insight:
The prioritization model appears effective because it identifies customers who are most likely to contribute to future losses based on their historical behavior. By focusing collections efforts on borrowers with multiple defaults, EduFin can allocate resources more efficiently and improve recovery outcomes. Repeated default behavior is often a stronger predictor of future risk than any single credit score or demographic characteristic.

### **6. Ideal High-Risk Borrower Profile (Final Synthesis)**

Based on all analyses (CIBIL, Age, Income, Employment, Behavior),  
describe the "typical high-risk borrower" EduFin should monitor closely.

Include:
- CIBIL range
- Age group
- Income bracket
- Employment type
- Loan/default behavior

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
The typical high-risk borrower profile is a young, self-employed customer with lower income levels, a weaker credit profile, and a history of repeated defaults. The analysis showed that younger age groups experienced higher default rates, while self-employed customers in the lowest income segment exhibited the highest default rate among employment-income combinations. Repeated default behavior further increases collection priority and overall risk exposure.
Business Insight:
EduFin should closely monitor borrowers who combine multiple risk factors, particularly younger self-employed individuals with lower incomes and prior default history. Enhanced underwriting, additional income verification, risk-based pricing, and proactive account monitoring could help reduce losses and improve portfolio performance.

## INVESTIGATION SUMMARY

**Write a 3–4 sentence executive summary synthesizing your complete customer risk analysis:**

**Example format:**

*"Customer risk analysis shows a clear inverse relationship between CIBIL score and default rates, with sub-650 segments contributing the highest risk. Younger borrowers (18–35) and low-income groups (<6L) exhibit higher default behavior, while certain profiles such as self-employed low-income customers emerge as the riskiest segment. Loss exposure is concentrated within specific income brackets, indicating both frequency and ticket size play a role in risk. A priority-based collections model identifies high-impact recovery targets, recommending immediate focus on high-default, low-CIBIL customers and stricter underwriting for high-risk demographic segments."*

---

**[Write your summary below - 3-4 sentences covering:**
- **WHO is defaulting (segments)**
- **WHAT patterns exist (CIBIL, age, income, employment)**
- **WHERE losses are concentrated**
- **WHAT actions should be taken (collections + underwriting)**]

- Final Investigation Summary
Customer risk analysis revealed that default behavior varies across multiple borrower characteristics, with younger borrowers and self-employed customers in the lowest income segment showing the highest default rates. While lower-income groups exhibited higher default percentages, the largest financial losses were concentrated in the 12L+ income segment, indicating that larger loan balances drive overall portfolio exposure. The collections priority model successfully identified high-risk customers based primarily on repeated default behavior, making them strong candidates for recovery efforts. EduFin should strengthen underwriting for high-risk segments, particularly low-income self-employed borrowers, while focusing collection resources on customers with recurring defaults and high loss potential.

%md
## WHY GATE - Business Reasoning

### Question 1: Aggregation Logic (Customer vs Loan Level)

Across your analysis (especially Query 2A–2E), you use both COUNT(DISTINCT customer_id) and COUNT(loan_id).  
Why is it important to distinguish between customer-level and loan-level aggregation?  
How does using COUNT(loan_id) ensure accurate calculation of default rates?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
It is important to distinguish between customer-level and loan-level aggregation because a single customer may have multiple loans. Using COUNT(DISTINCT customer_id) measures the number of unique borrowers, while COUNT(loan_id) measures the total number of loans. Since defaults occur at the loan level, using COUNT(loan_id) provides a more accurate calculation of default rates by capturing every loan and default event. This prevents underestimating risk when customers hold multiple loans with different repayment outcomes.
Business Insight:
Loan-level aggregation gives EduFin a more precise view of portfolio risk because it reflects the performance of each individual loan rather than treating all customers equally regardless of the number of loans they hold. This helps management make better lending, underwriting, and collections decisions based on actual exposure and default behavior.


### Question 2: Segment-Based Risk vs Individual Thresholds

In Query 2C, instead of filtering customers using a fixed income cutoff (e.g., income > ₹5L), you analyzed default behavior across income segments.

Why does segment-level analysis provide more reliable risk insights than using arbitrary thresholds?
How can segmentation reveal patterns that fixed cutoffs may hide?


**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
Segment-level analysis provides more reliable risk insights because it compares behavior across multiple income groups rather than separating customers into only two categories using a fixed cutoff. By analyzing segments such as <3L, 3L–6L, 6L–12L, and 12L+, patterns in both default rates and total losses become visible. A fixed threshold may hide important differences within groups and overlook segments that contribute significant risk despite falling on the same side of the cutoff. Segmentation revealed that lower-income groups had higher default rates, while the highest-income group generated the largest total default losses due to larger loan amounts.
Business Insight:
Segment-based analysis allows EduFin to tailor lending and risk management strategies to specific customer groups rather than applying a one-size-fits-all rule. This approach helps identify both high-frequency default segments and high-loss exposure segments, leading to more effective underwriting, pricing, and collections decisions. By understanding risk patterns across multiple income levels, the company can allocate resources more accurately and reduce overall portfolio risk.


### Question 3: Multi-Factor Risk Modeling (Beyond Single Variable)

Query 2E introduces a priority score combining multiple factors (default amount, CIBIL score, and default count).  
Why is a multi-factor scoring approach more effective than relying on a single metric like default amount alone?


**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
A multi-factor scoring approach is more effective because customer risk is influenced by several factors rather than a single metric. Query 2E combined default amount, CIBIL score, and default count to create a more complete assessment of borrower risk. For example, a customer with a large default amount may not be as risky as a customer with repeated defaults and a poor credit history. By evaluating multiple indicators together, the model identifies customers who represent the greatest overall risk to the portfolio.
Business Insight:
Using a multi-factor model allows EduFin to prioritize collections efforts more accurately and avoid relying on a single measure that may overlook important risk signals. This approach improves decision-making by balancing exposure, repayment behavior, and creditworthiness. As a result, collections teams can focus on high-impact recovery targets while management gains a more reliable view of portfolio risk.

### Question 4: Impact of Sample Size on Risk Insights

When comparing results from edufin_small and edufin_national, some default rate patterns may stabilize while others may change.

Why does increasing sample size improve statistical confidence in risk analysis?
What does it mean if a trend observed in a small dataset weakens or disappears in a larger dataset?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
Increasing sample size improves statistical confidence because larger datasets are more representative of the overall customer population and reduce the influence of random variation or outliers. Trends identified in a small dataset may change or disappear when tested on a larger dataset, indicating they were caused by limited sample size rather than a true business pattern. Consistent findings across both small and national datasets provide stronger evidence that the observed risk relationships are reliable. Business Insight: EduFin should validate lending and collections strategies using larger datasets whenever possible, since decisions based on statistically stable patterns are more likely to improve underwriting accuracy, reduce portfolio risk, and support long-term business performance.

### Question 5: Scale Validation (Small vs National Dataset)

If a relationship exists in edufin_small but disappears in edufin_national, what could that imply about sample bias, statistical stability, and data representativeness?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
Increasing sample size improves statistical confidence because larger datasets are more representative of the overall customer population and reduce the impact of random variation. Trends observed in a small dataset may be influenced by a few unusual customers or isolated events, while larger datasets help reveal patterns that are more consistent and reliable. If a trend weakens or disappears when analyzed in a larger dataset, it suggests that the original finding may have been caused by limited data rather than a true underlying risk pattern. Conversely, trends that remain consistent across both small and large datasets are more likely to represent genuine business risks.
Business Insight:
EduFin should place greater confidence in risk patterns that persist as sample size increases because they are more likely to generalize across the entire portfolio. Validating findings on larger datasets helps prevent decisions based on temporary fluctuations or outliers. This approach supports more accurate underwriting, pricing, and collections strategies while reducing the risk of acting on misleading conclusions.

## SUBMISSION READINESS CHECK

**Before submitting, verify:**

✓ Data Validation (Cell 2) passed

✓ All 5 SQL queries (Cells 3-7) executed without errors

✓ Executive Summary (Cell 8) - all 4 questions answered

✓ Investigation Summary (Cell 9) - 3-4 sentence synthesis written

✓ WHY Gate (Cell 10) - all 3 questions answered with business reasoning

✓ Formatting: Currency in Crores/Lakhs, percentages to 2 decimals

✓ Column aliases: Business-friendly names (not raw database column names)

✓ Results reviewed: Do the numbers make business sense?

---

## SUBMISSION PROCESS

**Step 1:** Run Cell 12 below
- Enter your name and select day
- Get your submission filename and form link

**Step 2:** Export this notebook as HTML
1. Go to **File** menu (top left)
2. Click on **Export**
3. Select **IPython notebook** format from the dropdown
4. Click **Export** button
5. Your browser will download the file

**Step 3:** Rename the downloaded file
- Use the exact filename shown in Cell 12 output
- Example: `Name_Batch_no_Day1.ipynb`

**Step 4:** Submit via Google Form
- Open the form link from Cell 12 output
- Upload your renamed ipynb file
- Note your entry number from form confirmation (tracking ID for feedback)

**Step 5:** Wait for feedback
- Expect feedback within 24 hours via WhatsApp

---

**Reminder:** Partial submissions accepted. If only 3 of 5 queries work, submit what you have with documentation of where you got stuck.

In [0]:
import re

FORM_LINK = "https://forms.gle/7xWVjeLRahSEMYXD9"  

dbutils.widgets.removeAll()
dbutils.widgets.text("Name", "")
dbutils.widgets.text("EnrollmentID", "")
dbutils.widgets.dropdown("day", "Day3", ["Day1", "Day2", "Day3"])

name = dbutils.widgets.get("Name")
batch = dbutils.widgets.get("EnrollmentID")
day = dbutils.widgets.get("day")

if not name or not batch:
    print("⚠️  Enter your name and BatchNo above, then run this cell")
else:
    clean_name = re.sub(r'[^a-zA-Z0-9]', '', name.lower())
    filename = f"{clean_name}_{batch}_{day}.ipynb"
    
    print("File -> Export -> IPython notebook")
    print(f"Rename to: {filename}")
    print(f"Submit the file in the form link: {FORM_LINK}")
    print("\nPlease await for submission review.")

File -> Export -> IPython notebook
Rename to: reneekammeyer_SAP320B12_Day2.ipynb
Submit the file in the form link: https://forms.gle/7xWVjeLRahSEMYXD9

Please await for submission review.
